# Ejercicio 2 — Efecto de añadir x2 bajo distintos niveles de colinealidad

**Introducción al Machine Learning — Cuarta lista de ejercicios — Daniel Vargas**

Se usan `experimento_2_a.csv`, `experimento_2_b.csv` y `experimento_2_c.csv`
(columnas `x1`, `x2`, `y`). Para cada archivo se ajustan `x1 -> y` y
`(x1,x2) -> y`, reutilizando las funciones de `src/lista4/`.

In [1]:
import numpy as np
import pandas as pd

from lista4.data import cargar_csv, obtener_xy
from lista4.models import ajustar_regresion_lineal
from lista4.metrics import riesgo_cuadratico, rango

archivos = {
    'a': 'experimento_2_a.csv',
    'b': 'experimento_2_b.csv',
    'c': 'experimento_2_c.csv',
}

dfs = {clave: cargar_csv(nombre) for clave, nombre in archivos.items()}
{clave: df.shape for clave, df in dfs.items()}

{'a': (250, 3), 'b': (250, 3), 'c': (250, 3)}

## (a)-(c) Correlación, riesgos y coeficientes por archivo, en una sola tabla

In [2]:
filas = []
for clave, df in dfs.items():
    X_x1, y = obtener_xy(df, 'x1', 'y')
    X_amb, _ = obtener_xy(df, ['x1', 'x2'], 'y')

    modelo_x1 = ajustar_regresion_lineal(X_x1, y)
    modelo_amb = ajustar_regresion_lineal(X_amb, y)

    R_x1 = riesgo_cuadratico(y, modelo_x1.predict(X_x1))
    R_amb = riesgo_cuadratico(y, modelo_amb.predict(X_amb))

    corr_x1x2 = np.corrcoef(df['x1'], df['x2'])[0, 1]
    beta1, beta2 = modelo_amb.coef_

    filas.append({
        'archivo': clave,
        'corr(x1,x2)': corr_x1x2,
        'R_x1': R_x1,
        'R_x1x2': R_amb,
        'beta1_hat': beta1,
        'beta2_hat': beta2,
    })

tabla_2 = pd.DataFrame(filas).set_index('archivo')
tabla_2

,"corr(x1,x2)",R_x1,R_x1x2,beta1_hat,beta2_hat
archivo,,,,,
a,0.999760,1.029350,1.025907,5.957231,-2.900650
b,0.999779,1.061539,1.059304,0.606315,2.325776
c,0.999821,1.141281,1.140041,4.903450,-1.911232


## (d) $\Delta R = \hat R_{x1} - \hat R_{x1,x2}$ por archivo

In [3]:
tabla_2['delta_R'] = tabla_2['R_x1'] - tabla_2['R_x1x2']
tabla_2[['R_x1', 'R_x1x2', 'delta_R']]

,R_x1,R_x1x2,delta_R
archivo,,,
a,1.029350,1.025907,0.003443
b,1.061539,1.059304,0.002235
c,1.141281,1.140041,0.001240


## (e) Rango de $\hat\beta_1$, $\hat\beta_2$ y $\Delta R$ entre los 3 archivos

In [4]:
rango_beta1 = rango(tabla_2['beta1_hat'])
rango_beta2 = rango(tabla_2['beta2_hat'])
rango_deltaR = rango(tabla_2['delta_R'])

print(f'rango(beta1_hat) = {rango_beta1:.4f}')
print(f'rango(beta2_hat) = {rango_beta2:.4f}')
print(f'rango(delta_R)   = {rango_deltaR:.6f}')

rango(beta1_hat) = 5.3509
rango(beta2_hat) = 5.2264
rango(delta_R)   = 0.002203


## (f) Interpretación

Comparando (d) y (e):

In [5]:
print(tabla_2)
print()
print('rango(delta_R)   =', rango_deltaR)
print('rango(beta1_hat) =', rango_beta1)
print('rango(beta2_hat) =', rango_beta2)

         corr(x1,x2)      R_x1    R_x1x2  beta1_hat  beta2_hat   delta_R
archivo                                                                 
a           0.999760  1.029350  1.025907   5.957231  -2.900650  0.003443
b           0.999779  1.061539  1.059304   0.606315   2.325776  0.002235
c           0.999821  1.141281  1.140041   4.903450  -1.911232  0.001240

rango(delta_R)   = 0.0022033146837794337
rango(beta1_hat) = 5.350916024405555
rango(beta2_hat) = 5.226425956586279


**Respuesta.** El valor de $\Delta R$ es pequeño (del orden de $10^{-3}$) y
muy similar en los tres archivos (rango bajo comparado con la escala del
riesgo, que es de orden 1), por lo que **incorporar x2 no produce un cambio
importante en el riesgo empírico** en ninguno de los tres casos: x1 ya
explica casi toda la variabilidad de y.

Los tres archivos tienen además una correlación `corr(x1,x2)` casi idéntica
entre sí y muy cercana a 1 (alrededor de 0.9998 en los tres casos). Sin
embargo, a pesar de que esa colinealidad es prácticamente la misma en los
tres archivos, $\hat\beta_1$ y $\hat\beta_2$ **no** cambian en la misma
proporción entre ellos: toman valores muy distintos, e incluso $\hat\beta_2$
cambia de signo entre archivos. Esto es el síntoma clásico de la
multicolinealidad: cuando dos predictores están muy correlacionados, el
modelo conjunto sigue prediciendo bien y el riesgo es estable, pero la
forma en que el ajuste "reparte" el efecto entre $\hat\beta_1$ y
$\hat\beta_2$ es inestable y muy sensible a variaciones pequeñas en los
datos. En resumen: **el riesgo es estable entre los tres archivos, pero los
coeficientes individuales no lo son**.